# Thirstwave detection

# Importing libraries and defining functions

In [10]:
import numpy as np
import xarray as xr
import pandas as pd
from pathlib import Path

def find_et0_file(model: str, exp: str, var: str, base_dir = Path("/work10/archive/CMIP6/CMIP-SSPs/outputs")) -> Path:
    """Find the ET0 (not ET0adv/ET0rad/VPD) file for a model/experiment.

    Uses a glob instead of a hardcoded date range because some models
    (e.g. UKESM1-0-LL, which runs a 360-day calendar) end their filenames
    in a different date than the rest (...1230 instead of ...1231).
    """
    matches = sorted(base_dir.glob(f"{model}/{exp}/{model}_{exp}_daily_{var}_*.nc"))
    if not matches:
        raise FileNotFoundError(f"No {var} file found in {base_dir} for {model}/{exp}")
    return matches[0]

def find_percentiles_file(model: str, base_dir = Path("/work10/archive/CMIP6/CMIP-SSPs/thirstwave_detection/90th_percentiles")) -> Path:
    """Find the ET0 90th percentile file for a model."""
    matches = sorted(base_dir.glob(f"{model}_historical_90th_percentiles.nc"))
    if not matches:
        raise FileNotFoundError(f"No percentiles file found in {base_dir} for {model}")
    return matches[0]

def preprocess_ET0(ds: xr.Dataset) -> xr.DataArray:
    """Loads daily ET0 for the specified model and experiment, converts to 365-day calendar and adds a 'dayofyear' coordinate."""   
    
    # Ensure all years have 365 days by removing Feb. 29th
    ds = ds.convert_calendar('noleap', align_on='date')
    
    # Add 'dayofyear' coordinate (1 = Jan. 1st, ... , 365 = Dec. 31st)
    ds = ds.assign_coords(dayofyear=ds.time.dt.dayofyear)
    
    # Safety check
    assert len(pd.unique(ds.dayofyear.values)) == 365

    return ds.ET0
    

def calculate_valid_event_days(et0: xr.DataArray, p90: xr.Dataset) -> xr.Dataset:
    """Returns a Dataset """
    
    # Days where ETos is above the threshold
    above = et0.groupby(et0.dayofyear) > p90
    
    # Identify >=3 consecutive days
    consecutive_3 = above.rolling(time=3, center=False).sum() == 3
    valid_event_days = (
        consecutive_3
        | consecutive_3.shift(time=-1, fill_value=False)
        | consecutive_3.shift(time=-2, fill_value=False)
    )
    return valid_event_days

def calculate_thirstwave_stats(valid_event_days: xr.Dataset, et0: xr.DataArray) -> xr.Dataset:
    """Measures thirstwave frequency, intensity and duration from the mask of valid event days
    (one value per grid cell and year) """

    # A day is the start of an event if it's flagged but the day before it wasn't
    event_starts = valid_event_days & ~valid_event_days.shift(time=1, fill_value=False)
    
    # Frequency = number of events each year (year⁻¹)
    frequency = event_starts.astype(int).groupby('time.year').sum('time')
    
    # Intensity = mean ETos anomaly above the 90th-percentile threshold during thirstwave days (mm/day)
    anomaly = et0.groupby(et0.dayofyear) - p90
    intensity = anomaly.where(valid_event_days).groupby('time.year').mean('time')
    
    # Duration = mean thirstwave event duration per year (days)
    event_days_per_year = valid_event_days.astype(int).groupby('time.year').sum('time')
    duration = event_days_per_year / frequency.where(frequency > 0)
    
    # Save metrics to file
    thirstwave_stats = xr.Dataset({'frequency': frequency.ET0,
                                   'duration': duration.ET0,
                                   'intensity': intensity.ET0
                                   })
    thirstwave_stats['frequency'].attrs = {'long_name': 'Number of thirstwave events per year',
                                           'units': 'count year-1'
                                           }
    thirstwave_stats['duration'].attrs = {'long_name': 'Mean thirstwave event duration per year',
                                          'units': 'days'}
    thirstwave_stats['intensity'].attrs = {'long_name': 'Mean ETos anomaly above 90th percentile during thirstwave days',
                                           'units': 'mm day-1'}
    return thirstwave_stats

In [ ]:
import cmip6_archive as ca

models = [m.name for m in ca.GCM_REGISTRY]
experiments = ["ssp126", "ssp245", "ssp370", "ssp585"]

for model in models:
    for exp in experiments:
        try:
            print(f"[{model} / {exp}] Identifying thirstwaves...")
            
            # Load and preprocess ET0 data
            file = find_et0_file(model, exp, var = 'ET0')
            ds = xr.open_dataset(file)
            et0 = preprocess_ET0(ds)

            # Load ETos 90th percentile for this model
            file_p90 = find_percentiles_file(model)
            p90 = xr.open_dataset(file_p90)
            
            # Calculate valid event days and save to a file
            valid_event_days = calculate_valid_event_days(et0, p90)
            outdir = Path("/work10/archive/CMIP6/CMIP-SSPs/thirstwave_detection/valid_event_days")
            years = valid_event_days.time.dt.year.values
            filename = outdir / f"{model}_{exp}_valid_event_days_{years[0]}-{years[-1]}.nc"
            valid_event_days.to_netcdf(filename, engine='netcdf4')
            print(f"[{model} / {exp}] Saving valid event days to {filename} ...")

            # Calculate annual thirswave frequency, magnitude and duration, and save to a file
            thirstwave_stats = calculate_thirstwave_stats(valid_event_days, et0)
            outdir_stats = Path("/work10/archive/CMIP6/CMIP-SSPs/thirstwave_detection/thirstwave_features")
            filename_stats = outdir_stats / f"{model}_{exp}_thirstwave_features_{years[0]}-{years[-1]}.nc"
            print(f"[{model} / {exp}] Saving thirstwave features to {filename_stats} ...")
            thirstwave_stats.to_netcdf(filename_stats, engine='netcdf4')
            
        except Exception as e:
            print(f"[{model} / {exp}] ERROR {e}")

[IPSL-CM6A-LR / ssp126] Saving valid event days to /work10/archive/CMIP6/CMIP-SSPs/thirstwave_detection/valid_event_days/IPSL-CM6A-LR_ssp126_valid_event_days_2015-2100.nc ...
[IPSL-CM6A-LR / ssp126] Saving thirstwave features to /work10/archive/CMIP6/CMIP-SSPs/thirstwave_detection/thirstwave_features/IPSL-CM6A-LR_ssp126_thirstwave_features_2015-2100.nc ...
[IPSL-CM6A-LR / ssp245] Saving valid event days to /work10/archive/CMIP6/CMIP-SSPs/thirstwave_detection/valid_event_days/IPSL-CM6A-LR_ssp245_valid_event_days_2015-2100.nc ...
[IPSL-CM6A-LR / ssp245] Saving thirstwave features to /work10/archive/CMIP6/CMIP-SSPs/thirstwave_detection/thirstwave_features/IPSL-CM6A-LR_ssp245_thirstwave_features_2015-2100.nc ...
[IPSL-CM6A-LR / ssp370] Saving valid event days to /work10/archive/CMIP6/CMIP-SSPs/thirstwave_detection/valid_event_days/IPSL-CM6A-LR_ssp370_valid_event_days_2015-2100.nc ...
[IPSL-CM6A-LR / ssp370] Saving thirstwave features to /work10/archive/CMIP6/CMIP-SSPs/thirstwave_detection/

# Sanity check

In [ ]:
import matplotlib.pyplot as plt
import scienceplots

YEAR = 2022

# Set plot style
plt.style.use('nature')
plt.rcParams.update({'figure.dpi': 300})
fig, ax = plt.subplots(figsize=(5, 3))

#TODO: load the datasets for one model and experiment here
ds = xr.open_datset(find_et0_file('MIROC6', 'ssp585', var = 'ET0'))
et0 = preprocess_ET0(ds)
p90 = xr.open_dataset(find_percentiles_file('MIROC6'))
valid_event_days = calculate_valid_event_days(et0, p90)

# Select ET0 and threshold values at grid cell (0, 0)
et0_plot = et0.sel(time=str(YEAR)).sel(lat=0, lon=0)
p90_plot = p90.sel(lat=0, lon=0).ET0
x = et0_plot.time

# Black line = ET0 values, Red line = threshold
ax.plot(x, et0_plot, color='black', label=r'$ET_0$')
ax.plot(x, p90_plot, color='red', label="90th percentile threshold")

# Mark thirswave days with an X
event_days_plot = valid_event_days.sel(time=str(YEAR)).sel(lat=0, lon=0)
days_in_event = x[event_days_plot.ET0.values].values
et0_value_in_event_days = et0_plot.sel(time=days_in_event)
ax.scatter(days_in_event, et0_value_in_event_days, zorder=2, s=6, label='Thirstwave days', marker='x', color='blue')

plt.suptitle(f"ET0 and thristwave days at (lat, lon) = (0, 0) during {YEAR}")
plt.legend()

In [ ]:
n_thirstwave_days = event_days_plot.ET0.values.sum()
print(f"There were {n_thirstwave_days} thirstwave days in {YEAR} at (lat, lon) = (0, 0)")

In [ ]:
fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(8,2))

freq = frequency.sel(year=YEAR).ET0
freq = freq.drop_attrs().assign_attrs({'long_name': 'Number of events'})
freq.plot(ax=axes[0], label='a', cmap='Blues')
axes[0].set_title(f"Thirstwave frequency ({YEAR})")

dur = duration.sel(year=YEAR).ET0
dur = dur.drop_attrs().assign_attrs({'long_name': 'Mean event duration'})
dur.plot(ax=axes[1], cmap='Oranges')
axes[1].set_title(f"Thirstwave duration ({YEAR})")